# 04b - Latent Capacity Sweep

Sweep dataset, training-network count, frames per network, latent dimension, and repeat. Training, persistence, restoration, rollout evaluation, and latent readouts all use the shared `lss.latent` workflow.


In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lss.latent.capacity import (
    build_capacity_specs,
    fit_initial_latent_readouts,
    run_capacity_sweep,
)
from lss.latent.experiment import seed_everything
from lss.utils import resolve_device

plt.rcParams['figure.dpi'] = 120


## Configuration


In [ ]:
seed = random.SystemRandom().randrange(1, 2**31)
seed_everything(seed)
print('run seed:', seed)

sweep_dataset_names = ['depablo-10k']
latent_dims = [1, 2, 4, 8]
train_counts = [1, 2, 5, 10, 20]
train_frame_counts = [25]
target_modes = ['delta']
sweep_repeats = 3

dataset_specs = {
    'reid': {
        'label': 'Reid',
        'path': '../../data/new_reid_combined.pt',
        'val_count': 30,
    },
    'depablo': {
        'label': 'dePablo OOL',
        'path': '../../data/2340_dePablo_networks_OOL_undirected.pt',
        'val_count': 30,
    },
    'depablo-10k': {
        'label': 'dePablo 10k',
        'path': '../../data/depablo-10k.pt',
        'val_count': 30,
    },
}

cfg = {
    'split_seed': seed,
    'min_train_p_ratio': None,
    'device': 'auto',
    'pos_dim': 2,
    'batch_graphs': 8,
    'frame_skip': 1,
    'train_frame_start_order': 1,
    'latent_tokens': 12,
    'hidden_size': 90,
    'edge_feature_dim': 8,
    'ae_max_epochs': 250,
    'ae_patience': 3,
    'ae_lr': 3e-4,
    'ae_weight_decay': 1e-5,
    'dyn_max_epochs': 250,
    'dyn_patience': 3,
    'dyn_lr': 1e-4,
    'dyn_weight_decay': 1e-5,
    'propagator_loss': 'delta',
    'propagator_objective': 'one_step',
    'propagator_step_stride': 1,
    'initial_velocity': 'zero',
    'propagator_model': None,
    'early_stop_min_delta': 1e-5,
    'rollout_steps_grid': list(range(10, 301, 10)),
    'temperature_pratio_window': 'full',
    'output_root': '../results/latent_space_capacity_sweep/full_sweep',
}

run_dir = Path(cfg['output_root'])
device = resolve_device(cfg['device'])
sweep_specs = build_capacity_specs(
    sweep_dataset_names,
    dataset_specs,
    cfg,
    latent_dims=latent_dims,
    train_counts=train_counts,
    train_frame_counts=train_frame_counts,
    target_modes=target_modes,
    repeats=sweep_repeats,
    seed=seed,
)
print('device:', device)
print('runs:', len(sweep_specs))
print('output:', run_dir)


## Train Or Load Sweep


In [ ]:
sweep_tables = run_capacity_sweep(
    sweep_specs,
    cfg,
    device=device,
    output_dir=run_dir,
    force_training=False,
)

sweep_summary_df = sweep_tables['summary']
sweep_summary_mean_df = sweep_tables['summary_mean']
sweep_rollout_stats_df = sweep_tables['rollout_stats']
sweep_rollout_raw_df = sweep_tables['rollout_raw']
sweep_ae_history_df = sweep_tables['ae_history']
sweep_dyn_history_df = sweep_tables['dyn_history']
sweep_model_manifest_df = sweep_tables['model_manifest']
latent_pratio_raw_df = sweep_tables['latent_raw']

display(sweep_summary_mean_df.round(5))


## Initial Latent To Final P-Ratio


In [ ]:
initial_latent_selection_df, initial_latent_prediction_df, initial_latent_summary_df = fit_initial_latent_readouts(
    latent_pratio_raw_df,
    ridge_alpha=1e-3,
)

export_dir = run_dir / 'exports'
initial_latent_selection_df.to_csv(export_dir / 'initial_latent_pratio_readout_by_run.csv', index=False)
initial_latent_prediction_df.to_csv(export_dir / 'initial_latent_pratio_predictions.csv', index=False)
initial_latent_summary_df.to_csv(export_dir / 'initial_latent_pratio_readout_summary.csv', index=False)
display(initial_latent_summary_df.round(4))


## Sweep Summary


In [ ]:
rollout_summary = (
    sweep_rollout_stats_df[sweep_rollout_stats_df['split'].eq('test')]
    .groupby(
        ['dataset_name', 'latent_dim', 'train_networks', 'train_frames_per_network', 'rollout_steps'],
        as_index=False,
    )
    .agg(
        p_ratio_r2_mean=('p_ratio_r2', 'mean'),
        p_ratio_r2_std=('p_ratio_r2', 'std'),
        position_mse_mean=('final_pos_mse', 'mean'),
        position_mse_std=('final_pos_mse', 'std'),
        n_repeats=('repeat_idx', 'nunique'),
    )
)
rollout_summary[['p_ratio_r2_std', 'position_mse_std']] = rollout_summary[
    ['p_ratio_r2_std', 'position_mse_std']
].fillna(0.0)
rollout_summary.to_csv(export_dir / 'rollout_summary.csv', index=False)

for dataset_name, dataset_df in rollout_summary.groupby('dataset_name'):
    final_step = int(dataset_df['rollout_steps'].max())
    final_df = dataset_df[dataset_df['rollout_steps'].eq(final_step)]
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
    for (networks, frames), group in final_df.groupby(['train_networks', 'train_frames_per_network']):
        group = group.sort_values('latent_dim')
        label = f'{int(networks)} networks, {int(frames)} frames'
        axes[0].errorbar(
            group['latent_dim'], group['p_ratio_r2_mean'], yerr=group['p_ratio_r2_std'],
            marker='o', capsize=3, label=label,
        )
        axes[1].errorbar(
            group['latent_dim'], group['position_mse_mean'], yerr=group['position_mse_std'],
            marker='o', capsize=3, label=label,
        )
    axes[0].set(title=f'{dataset_name}: p-ratio rollout at {final_step} frames', xlabel='latent dimension', ylabel='test R2')
    axes[0].axhline(0, color='0.25', lw=1, ls='--')
    axes[1].set(title=f'{dataset_name}: position error at {final_step} frames', xlabel='latent dimension', ylabel='test position MSE', yscale='log')
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(frameon=False, fontsize=7)
    plt.show()

successful_readouts = initial_latent_selection_df[
    initial_latent_selection_df['combo_test_r2'].ge(0.70)
].sort_values('combo_test_r2', ascending=False)
successful_rollouts = sweep_rollout_stats_df[
    sweep_rollout_stats_df['split'].eq('test')
    & sweep_rollout_stats_df['p_ratio_r2'].ge(0.70)
].sort_values('p_ratio_r2', ascending=False)

print('successful initial latent readouts (test R2 >= 0.70)')
display(successful_readouts[[
    'dataset_name', 'latent_dim', 'train_networks', 'train_frames_per_network',
    'repeat_idx', 'combo_test_r2',
]].round(4))
print('successful autoregressive rollouts (test R2 >= 0.70)')
display(successful_rollouts[[
    'dataset_name', 'latent_dim', 'train_networks', 'train_frames_per_network',
    'repeat_idx', 'rollout_steps', 'p_ratio_r2', 'final_pos_mse',
]].round(5))
